# Notebook 04 — ML Model Training & Evaluation

**Predicting the Success Probability of DJ, BV, and Simon's Algorithms Under Noise**

---

## Objective
In this notebook, we train and evaluate three regression models to predict success probability:

1. **Baseline** — Linear Regression on `circuit_depth` alone (simple reference)
2. **Decision Tree** — Tuned with GridSearchCV + 5-fold cross-validation
3. **Random Forest** — Tuned with GridSearchCV + 5-fold cross-validation

### Evaluation metrics:
- **R² (coefficient of determination)** — How much variance the model explains (1.0 = perfect)
- **MAE (mean absolute error)** — Average prediction error
- **RMSE (root mean squared error)** — Penalizes large errors more heavily

## 4.1 Imports and Data Preparation

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import load_dataset, encode_features
from src.ml_pipeline import (
    split_data,
    train_baseline,
    train_decision_tree,
    train_random_forest,
    evaluate_model,
    compare_models,
    plot_predicted_vs_actual,
    plot_model_comparison,
    save_model,
)

sns.set_theme(style='whitegrid', palette='deep')
%matplotlib inline

print('All imports successful!')

In [ ]:
# Load and encode the dataset
data_path = os.path.join(project_root, 'data', 'results.csv')
df = load_dataset(data_path)

X, y, feature_names = encode_features(df)
print(f'\nFeatures: {len(feature_names)} columns')
print(f'Samples:  {len(X)}')

In [ ]:
# Train/test split (80/20)
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

---

## 4.2 Baseline Model

Our baseline is a **simple linear regression** that uses only `circuit_depth` as a predictor. This establishes a floor — our tuned models must beat this to demonstrate genuine added value.

In [ ]:
# Train baseline (linear regression on circuit_depth only)
baseline_model = train_baseline(X_train, y_train)

In [ ]:
# Evaluate baseline
baseline_results = evaluate_model(
    baseline_model, X_test, y_test,
    model_name='Baseline (Linear Reg.)',
    depth_only=True
)

---

## 4.3 Decision Tree Regressor

A **Decision Tree** makes predictions through a series of yes/no splits on feature values. We tune its hyperparameters using **GridSearchCV** with 5-fold cross-validation.

### Hyperparameter grid:
| Parameter | Values | Meaning |
|---|---|---|
| `max_depth` | 3, 5, 8, 10, 15, None | Maximum tree depth |
| `min_samples_split` | 2, 5, 10 | Minimum samples to split a node |
| `min_samples_leaf` | 1, 2, 5 | Minimum samples in a leaf |

Total combinations: 6 × 3 × 3 = **54**, each evaluated with 5-fold CV.

In [ ]:
# Train Decision Tree with GridSearchCV
dt_model = train_decision_tree(X_train, y_train, cv=5)

In [ ]:
# Evaluate Decision Tree
dt_results = evaluate_model(
    dt_model, X_test, y_test,
    model_name='Decision Tree'
)

In [ ]:
# Predicted vs Actual plot for Decision Tree
plot_predicted_vs_actual(
    y_test, dt_results['y_pred'],
    model_name='Decision Tree',
    save_path='../outputs/figures/dt_predicted_vs_actual.png'
)

---

## 4.4 Random Forest Regressor

A **Random Forest** is an ensemble of many decision trees, each trained on a randomly sampled subset of the data and features. Their predictions are averaged for a more robust result.

### Hyperparameter grid:
| Parameter | Values | Meaning |
|---|---|---|
| `n_estimators` | 50, 100, 200 | Number of trees in the forest |
| `max_depth` | 5, 10, 15, None | Maximum tree depth |
| `min_samples_split` | 2, 5 | Minimum samples to split |
| `min_samples_leaf` | 1, 2 | Minimum samples in a leaf |

Total combinations: 3 × 4 × 2 × 2 = **48**, each with 5-fold CV.

In [ ]:
# Train Random Forest with GridSearchCV
rf_model = train_random_forest(X_train, y_train, cv=5)

In [ ]:
# Evaluate Random Forest
rf_results = evaluate_model(
    rf_model, X_test, y_test,
    model_name='Random Forest'
)

In [ ]:
# Predicted vs Actual plot for Random Forest
plot_predicted_vs_actual(
    y_test, rf_results['y_pred'],
    model_name='Random Forest',
    save_path='../outputs/figures/rf_predicted_vs_actual.png'
)

---

## 4.5 Model Comparison

Now let's compare all three models side by side.

In [ ]:
# Side-by-side comparison
all_results = [baseline_results, dt_results, rf_results]
comparison_df = compare_models(all_results)

In [ ]:
# Visual comparison
plot_model_comparison(
    comparison_df,
    save_path='../outputs/figures/model_comparison.png'
)

In [ ]:
# Calculate improvement over baseline
baseline_r2 = baseline_results['r2']
dt_r2 = dt_results['r2']
rf_r2 = rf_results['r2']

print('\nImprovement over Baseline:')
print(f'  Decision Tree: R² improved by {dt_r2 - baseline_r2:+.4f} '
      f'({(dt_r2 - baseline_r2) / max(abs(baseline_r2), 0.001) * 100:+.1f}%)')
print(f'  Random Forest: R² improved by {rf_r2 - baseline_r2:+.4f} '
      f'({(rf_r2 - baseline_r2) / max(abs(baseline_r2), 0.001) * 100:+.1f}%)')

## 4.6 Predicted vs Actual — All Models

In [ ]:
# Side-by-side predicted vs actual for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_data = [
    ('Baseline (Linear Reg.)', baseline_results),
    ('Decision Tree', dt_results),
    ('Random Forest', rf_results),
]

for ax, (name, results) in zip(axes, models_data):
    ax.scatter(y_test, results['y_pred'], alpha=0.3, s=15, color='#4C72B0',
               edgecolor='white', linewidth=0.2)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2)
    ax.set_xlabel('Actual', fontsize=11)
    ax.set_ylabel('Predicted', fontsize=11)
    ax.set_title(f'{name}\nR²={results["r2"]:.4f}', fontsize=13, fontweight='bold')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('Predicted vs Actual — All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/all_models_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Save Trained Models

In [ ]:
# Save all trained models
save_model(baseline_model, '../outputs/models/baseline_model.joblib')
save_model(dt_model, '../outputs/models/decision_tree_model.joblib')
save_model(rf_model, '../outputs/models/random_forest_model.joblib')

# Also save the comparison results
comparison_df.to_csv('../outputs/model_comparison.csv', index=False)
print('\nModel comparison saved to outputs/model_comparison.csv')

---

## Summary

In this notebook we:

1. **Split** the dataset into 80% training and 20% testing
2. **Trained** a baseline (linear regression on depth only) to establish a reference point
3. **Tuned** a Decision Tree with GridSearchCV (54 combinations × 5-fold CV)
4. **Tuned** a Random Forest with GridSearchCV (48 combinations × 5-fold CV)
5. **Compared** all models — tree-based models significantly outperform the baseline
6. **Saved** trained models for later use

**Key finding:** The tuned tree-based models capture the non-linear relationship between circuit/noise parameters and success probability far better than a simple linear baseline.

**Next:** In Notebook 05, we analyze feature importance and produce the final results.